<a href="https://colab.research.google.com/github/mjgpinheiro/Physics_models/blob/main/make_fig2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""Fig. 2: the three-level story. Panels (a) flux vs estimate, (b) Sabra
against its phase-randomised null, (c) Haar bands of i.i.d. returns against
their permutation construction null."""
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from ratio_estimator import causal_haar_bands, antisym

d = np.load("e9_panels.npy", allow_pickle=True).item()
Pi, v, null, per = d["Pi"], d["v"], d["null"], d["per"]
BLUE, ORANGE, GREY = "#215A8E", "#C75B39", "0.55"

# panel (c): i.i.d. returns through the Haar bands, and permutation nulls
rng = np.random.default_rng(11)
T = 200_000
adj = lambda G: np.array([G[i+1, i] for i in range(G.shape[0]-1)])
def band_op(r):
    B = causal_haar_bands(r)
    lc = lambda l: B[:-l].T @ B[l:] / (len(B)-l)
    return adj(antisym(np.linalg.solve(lc(1), lc(2)).T))
r = rng.standard_normal(T + 256)
obs_c = band_op(r)
null_c = np.array([band_op(rng.permutation(r)) for _ in range(40)])

fig, axes = plt.subplots(1, 3, figsize=(7.1, 2.5))
m = np.arange(len(Pi)) + 0.5

ax = axes[0]
ax.plot(m, Pi/Pi.max(), "o-", color=GREY, ms=3.5, lw=1, label=r"flux $\Pi_m$")
ax.plot(m, v/np.abs(v).max(), "s-", color=BLUE, ms=3.5, lw=1,
        label=r"$\hat G_{\rm as}$")
ax.axhline(0, color="k", lw=.7)
ax.set(xlabel="cut $m$", ylabel="normalised", title="(a) Sabra: sign, not amplitude")
ax.legend(frameon=False, fontsize=6.5, loc="center left", bbox_to_anchor=(0.02, 0.62))

ax = axes[1]
ax.hist(null, bins=28, color=GREY, edgecolor="none")
ax.axvline(v.mean(), color=BLUE, lw=1.6)
ax.annotate("observed", (v.mean(), ax.get_ylim()[1]*.78), color=BLUE,
            fontsize=6.5, ha="right", xytext=(-4, 0), textcoords="offset points")
ax.set(xlabel="mean forward score", ylabel="surrogates",
       title="(b) Sabra vs phase null")

ax = axes[2]
x = np.arange(4)
ax.bar(x-.19, obs_c, .38, color=BLUE, label="i.i.d. input")
ax.bar(x+.19, null_c.mean(0), .38, facecolor="white", edgecolor=ORANGE, lw=1.3,
       label="construction null")
ax.errorbar(x+.19, null_c.mean(0), yerr=2*null_c.std(0, ddof=1), fmt="none",
            ecolor=ORANGE, capsize=2.5, lw=.9)
ax.set_xticks(x, ["8→16", "16→32", "32→64", "64→256"], fontsize=6)
ax.set(xlabel="band (min)", ylabel="adjacent entry",
       title="(c) filters: all of it is geometry")
ax.legend(frameon=False, fontsize=6.5)

for a in axes:
    a.tick_params(labelsize=6.5)
    a.title.set_size(7.5); a.xaxis.label.set_size(7); a.yaxis.label.set_size(7)
fig.tight_layout(pad=0.4, w_pad=1.2)
fig.savefig("fig2_three_levels.pdf"); fig.savefig("fig2_three_levels.png", dpi=190)
print("observed(c):", np.round(obs_c,5), " null:", np.round(null_c.mean(0),5))


ModuleNotFoundError: No module named 'ratio_estimator'